# Adaptive v2 Kaggle GPU Run

This notebook pulls the versioned `adaptive_v2` code, verifies the Kaggle GPU, runs one seed for 50 rounds, and saves a local output log, JSON results, and per-family charts.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

project_dir = Path("/kaggle/working/mastercard_hackathon")
repository_url = "https://github.com/keshav-0210/mastercard_hackathon.git"
if not project_dir.exists():
    subprocess.run(["git", "clone", repository_url, str(project_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(project_dir), "pull", "--ff-only"], check=True)

sys.path.insert(0, str(project_dir / "src"))
os.chdir(project_dir)
os.environ["RUN_MODE"] = "KAGGLE_GPU"

import torch
print("Project:", project_dir)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
assert torch.cuda.is_available(), "Enable GPU in Kaggle before running this notebook"
print("Kaggle GPU check: OK")

In [ ]:
import tempfile
from datetime import datetime, timezone

from mastercard_defence.agents import QwenAgents
from mastercard_defence.llm import SharedLocalLLM
from mastercard_defence.loop import ClosedLoop, load_config

config = load_config(str(project_dir / "config" / "default.yaml"))
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
config["paths"]["memory_db"] = str(project_dir / "artifacts" / f"adaptive_v2_memory_{run_stamp}.sqlite")
config["generator_backend"] = "ctgan"
config["detector_mode"] = "static"
config["pipeline"]["rounds"] = max(50, int(config["pipeline"].get("rounds", 50)))
config["pipeline"]["synthetic_transactions"] = 400
config["pipeline"]["max_generated_attacks"] = 80

agents = QwenAgents(config, llm=SharedLocalLLM(config))
loop = ClosedLoop(config, agents=agents)
print("Adaptive v2 configuration ready")
print("Agent backend:", type(agents).__name__)
print("Generator:", config["generator_backend"])
print("Detector:", config["detector_mode"])
print("Seeds: 1")
print("Rounds:", config["pipeline"]["rounds"])
print("Run timestamp:", run_stamp)

In [ ]:
import contextlib
import io

class Tee(io.TextIOBase):
    def __init__(self, *streams):
        self.streams = streams
    def write(self, text):
        for stream in self.streams:
            stream.write(text)
            stream.flush()
        return len(text)
    def flush(self):
        for stream in self.streams:
            stream.flush()

artifacts_dir = project_dir / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)
log_path = artifacts_dir / f"adaptive_v2_run_{run_stamp}.log"

with log_path.open("w", encoding="utf-8") as log_file, contextlib.redirect_stdout(Tee(sys.stdout, log_file)), contextlib.redirect_stderr(Tee(sys.stderr, log_file)):
    print(f"LOG_SAVED {log_path}")
    print("RUN_CONFIGURATION seeds=1 rounds=50 generator=conditional_ctgan detector=static")
    try:
        suite = loop.run_robustness_suite(seeds=1, rounds=50)
    finally:
        loop.close()

print("One-seed adaptive v2 run completed")
print("Log:", log_path)

In [ ]:
def to_jsonable(value):
    if hasattr(value, "model_dump"):
        return to_jsonable(value.model_dump())
    if isinstance(value, dict):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(item) for item in value]
    return value

artifact = {
    "run_timestamp_utc": run_stamp,
    "experiment": "adaptive_v2_qwen_ctgan_static_detector",
    "agent_backend": type(agents).__name__,
    "generator_backend": "conditional_ctgan",
    "detector_mode": "static",
    "seed_count": suite["seed_count"],
    "rounds": suite["rounds"],
    "summary": suite["summary"],
    "by_seed": suite["by_seed"],
}
result_path = artifacts_dir / f"adaptive_v2_results_{run_stamp}.json"
result_path.write_text(json.dumps(to_jsonable(artifact), indent=2), encoding="utf-8")

sys.path.insert(0, str(project_dir))
from adaptive.generate_family_charts import build_charts
build_charts(result_path, project_dir / "adaptive" / "charts")

print("RESULTS_SAVED", result_path)
print("LOG_SAVED", log_path)
print("CHARTS_SAVED", project_dir / "adaptive" / "charts")
print("ADAPTIVE_V2_KAGGLE_RUN_OK", suite["seed_count"], suite["rounds"])